# ***Text Search Engine using TF-IDF***

### **Student name : Haitham Hani Dalloul**
### **Course name : Information Retrieval ITAP3312**
### **24/1/2026**

# **Project Overview**
**The project aim to use TF-IDF and Cosine Similarity concepts to bring the best result of search to the user query from a set of documets**

## **Libraries**

In [268]:
import pandas as pd # to deal with data fram
import numpy as np # for cosine similarity
import nltk # important for Tokenization، Stop words، Stemming
import string # to remove punctuation
import re # to remove digits , special characters
from sklearn.datasets import fetch_20newsgroups # to use the data set


## **Loading the Dataset**

In [269]:
from sklearn.datasets import fetch_20newsgroups # call the dataset from sklearn
# make an object to assign the data to it ,all type of massages and not removing anything from the data
newsgroups = fetch_20newsgroups(subset='all', remove=())
# list to handle the text
rows = []
# for loop to take the data from the data set and store it in csv form and every doc have a text and id
for i, text in enumerate(newsgroups.data):
    #it takes the file name without the source (folders)
    doc_id = newsgroups.filenames[i].split("/")[-1].split(".")[0]
    # change every category number to its corresponding category name
    group = newsgroups.target_names[newsgroups.target[i]]
    # add category name, document ID, and text to the rows list
    rows.append({
        "newsgroup": group,
        "document_id": doc_id,
        "text": text
    })
# change the array into data fram
IR_df = pd.DataFrame(rows)
# store it into csv form
IR_df.to_csv("20newsgroups_with_real_id.csv", index=False, encoding="utf-8")
# take the first 30 rows
df_subset = IR_df.head(30)



In [270]:
#from google.colab import drive
#drive.mount('/content/drive')
#IR_df = pd.read_csv('/content/drive/MyDrive/semastar 1 2025/Reports/IRDOC.csv')

#display first 5 rows
df_subset.head()



,newsgroup,document_id,text
0,rec.sport.hockey,54367,From: Mamatha Devineni Ratnam <mr47+@andrew.cm...
1,comp.sys.ibm.pc.hardware,60215,From: mblawson@midway.ecn.uoknor.edu (Matthew ...
2,talk.politics.mideast,76120,From: hilmi-er@dsv.su.se (Hilmi Eren)\nSubject...
3,comp.sys.ibm.pc.hardware,60771,From: guyd@austin.ibm.com (Guy Dawson)\nSubjec...
4,comp.sys.mac.hardware,51882,From: Alexander Samuel McDiarmid <am2o+@andrew...


In [271]:
#display last 5 rows
df_subset.tail()

,newsgroup,document_id,text
25,sci.space,60903,From: prb@access.digex.com (Pat)\nSubject: Con...
26,talk.politics.guns,54726,Subject: Re: BD's did themselves--you're all p...
27,comp.graphics,38295,From: Geoffrey_Hansen@mindlink.bc.ca (Geoffrey...
28,misc.forsale,76833,From: jtobias@cs.tamu.edu (Jason T Tobias)\nSu...
29,alt.atheism,54189,Organization: Penn State University\nFrom: Joh...


In [272]:
#display information about data column name ,datatype
df_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   newsgroup    30 non-null     object
 1   document_id  30 non-null     object
 2   text         30 non-null     object
dtypes: object(3)
memory usage: 852.0+ bytes


In [273]:
#df_subset.columns

## **Text Preprocessing**

1. Tokenization :split the text into words
2. lowercasing: make all the text in lower case later because the TF-IDF model will consider upper and lower words as different
3. Stop words: remove them because they can reduce the accuracy of results
4. punctuation: remove them because they do not add meaning to the text
5. digits: delete numbers because they are not important in the research
6. remove any special signs
7. Stemming: make the words in their root
8. white space removal
9. join text togthers




## **Example**
before: "Machine Learning is #1 in 2025!"

after : preprocessing : "machine learn"

# **Tokenization**

In [274]:
# download punkt and punkt_tab packages for NLTK tokenization
nltk.download('punkt')
nltk.download('punkt_tab')
#  import word_tokenize  to split text into words
from nltk.tokenize import word_tokenize
# assign the text column form datafram  and ensure that the data is string type, and convert it to a NumPy array for easier processing
text=df_subset['text'].astype(str).values
# for each document Tokenize  it into a list of words
tokens_text=[word_tokenize(t) for t in text]
#tokens_text[:]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# **Lowercasing**

In [275]:

# Convert all words in each document to lowercase
lower_text = [[word.lower() for word in doc_tokens] for doc_tokens in tokens_text]
#lower_text[0]


## **Stop words**

In [276]:
nltk.download('stopwords') # from nltk download stopword to deal withthem
# call  stopwords from NLTK
from nltk.corpus import stopwords
# Define the language for stopwords
stop_words = set(stopwords.words('english'))
# list to store documents after stopword removal
stop_word_text_del=[]
# for each words in the documnt are stop words remove it and store the cleaned words
for doc_tokens in lower_text:
    stop_word_text_del.append([word for word in doc_tokens if word not in stop_words])
stop_word_text_del[0]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


[':',
 'mamatha',
 'devineni',
 'ratnam',
 '<',
 'mr47+',
 '@',
 'andrew.cmu.edu',
 '>',
 'subject',
 ':',
 'pens',
 'fans',
 'reactions',
 'organization',
 ':',
 'post',
 'office',
 ',',
 'carnegie',
 'mellon',
 ',',
 'pittsburgh',
 ',',
 'pa',
 'lines',
 ':',
 '12',
 'nntp-posting-host',
 ':',
 'po4.andrew.cmu.edu',
 'sure',
 'bashers',
 'pens',
 'fans',
 'pretty',
 'confused',
 'lack',
 'kind',
 'posts',
 'recent',
 'pens',
 'massacre',
 'devils',
 '.',
 'actually',
 ',',
 'bit',
 'puzzled',
 'bit',
 'relieved',
 '.',
 'however',
 ',',
 'going',
 'put',
 'end',
 'non-pittsburghers',
 "'",
 'relief',
 'bit',
 'praise',
 'pens',
 '.',
 'man',
 ',',
 'killing',
 'devils',
 'worse',
 'thought',
 '.',
 'jagr',
 'showed',
 'much',
 'better',
 'regular',
 'season',
 'stats',
 '.',
 'also',
 'lot',
 'fo',
 'fun',
 'watch',
 'playoffs',
 '.',
 'bowman',
 'let',
 'jagr',
 'lot',
 'fun',
 'next',
 'couple',
 'games',
 'since',
 'pens',
 'going',
 'beat',
 'pulp',
 'jersey',
 'anyway',
 '.',
 '

# **Removing Punctuation and digits**

In [277]:
# Define punctuation characters to remove
punctuation_text=string.punctuation
# Remove punctuation, standalone digits, and words containing digits from each document
punctuation_text_del= [[word for word in doc_tokens if word not in punctuation_text and not word.isdigit()  and not any (num_word.isdigit() for num_word in word)]for doc_tokens in lower_text]
punctuation_text_del[0]

['from',
 'mamatha',
 'devineni',
 'ratnam',
 'andrew.cmu.edu',
 'subject',
 'pens',
 'fans',
 'reactions',
 'organization',
 'post',
 'office',
 'carnegie',
 'mellon',
 'pittsburgh',
 'pa',
 'lines',
 'nntp-posting-host',
 'i',
 'am',
 'sure',
 'some',
 'bashers',
 'of',
 'pens',
 'fans',
 'are',
 'pretty',
 'confused',
 'about',
 'the',
 'lack',
 'of',
 'any',
 'kind',
 'of',
 'posts',
 'about',
 'the',
 'recent',
 'pens',
 'massacre',
 'of',
 'the',
 'devils',
 'actually',
 'i',
 'am',
 'bit',
 'puzzled',
 'too',
 'and',
 'a',
 'bit',
 'relieved',
 'however',
 'i',
 'am',
 'going',
 'to',
 'put',
 'an',
 'end',
 'to',
 'non-pittsburghers',
 'relief',
 'with',
 'a',
 'bit',
 'of',
 'praise',
 'for',
 'the',
 'pens',
 'man',
 'they',
 'are',
 'killing',
 'those',
 'devils',
 'worse',
 'than',
 'i',
 'thought',
 'jagr',
 'just',
 'showed',
 'you',
 'why',
 'he',
 'is',
 'much',
 'better',
 'than',
 'his',
 'regular',
 'season',
 'stats',
 'he',
 'is',
 'also',
 'a',
 'lot',
 'fo',
 'fu

## **remove any special signs**

In [278]:
# removing any special characters  keeping only English letters (A-Z, a-z) and spaces
remove_signs = [[re.sub(r'[^a-zA-Z\s]', '', word) for word in doc_tokens] for doc_tokens in punctuation_text_del]
remove_signs[0]

['from',
 'mamatha',
 'devineni',
 'ratnam',
 'andrewcmuedu',
 'subject',
 'pens',
 'fans',
 'reactions',
 'organization',
 'post',
 'office',
 'carnegie',
 'mellon',
 'pittsburgh',
 'pa',
 'lines',
 'nntppostinghost',
 'i',
 'am',
 'sure',
 'some',
 'bashers',
 'of',
 'pens',
 'fans',
 'are',
 'pretty',
 'confused',
 'about',
 'the',
 'lack',
 'of',
 'any',
 'kind',
 'of',
 'posts',
 'about',
 'the',
 'recent',
 'pens',
 'massacre',
 'of',
 'the',
 'devils',
 'actually',
 'i',
 'am',
 'bit',
 'puzzled',
 'too',
 'and',
 'a',
 'bit',
 'relieved',
 'however',
 'i',
 'am',
 'going',
 'to',
 'put',
 'an',
 'end',
 'to',
 'nonpittsburghers',
 'relief',
 'with',
 'a',
 'bit',
 'of',
 'praise',
 'for',
 'the',
 'pens',
 'man',
 'they',
 'are',
 'killing',
 'those',
 'devils',
 'worse',
 'than',
 'i',
 'thought',
 'jagr',
 'just',
 'showed',
 'you',
 'why',
 'he',
 'is',
 'much',
 'better',
 'than',
 'his',
 'regular',
 'season',
 'stats',
 'he',
 'is',
 'also',
 'a',
 'lot',
 'fo',
 'fun',
 

## **Stemming**

In [279]:
# call  PorterStemmer class from NLTK
from nltk.stem import PorterStemmer
#Create a PorterStemmer object
stemmer = PorterStemmer()
# list to  stemmed words
stemming_text=[]
# for every word in the document bring the words to its root
for doc_tokens in remove_signs:
  stemming_text.append([stemmer.stem(word) for word in doc_tokens])

stemming_text[0]


['from',
 'mamatha',
 'devineni',
 'ratnam',
 'andrewcmuedu',
 'subject',
 'pen',
 'fan',
 'reaction',
 'organ',
 'post',
 'offic',
 'carnegi',
 'mellon',
 'pittsburgh',
 'pa',
 'line',
 'nntppostinghost',
 'i',
 'am',
 'sure',
 'some',
 'basher',
 'of',
 'pen',
 'fan',
 'are',
 'pretti',
 'confus',
 'about',
 'the',
 'lack',
 'of',
 'ani',
 'kind',
 'of',
 'post',
 'about',
 'the',
 'recent',
 'pen',
 'massacr',
 'of',
 'the',
 'devil',
 'actual',
 'i',
 'am',
 'bit',
 'puzzl',
 'too',
 'and',
 'a',
 'bit',
 'reliev',
 'howev',
 'i',
 'am',
 'go',
 'to',
 'put',
 'an',
 'end',
 'to',
 'nonpittsburgh',
 'relief',
 'with',
 'a',
 'bit',
 'of',
 'prais',
 'for',
 'the',
 'pen',
 'man',
 'they',
 'are',
 'kill',
 'those',
 'devil',
 'wors',
 'than',
 'i',
 'thought',
 'jagr',
 'just',
 'show',
 'you',
 'whi',
 'he',
 'is',
 'much',
 'better',
 'than',
 'hi',
 'regular',
 'season',
 'stat',
 'he',
 'is',
 'also',
 'a',
 'lot',
 'fo',
 'fun',
 'to',
 'watch',
 'in',
 'the',
 'playoff',
 'bo

## **join words**

In [280]:
# bring everything back as before by concatinat the words togther to make text
join_clean_text=[' '.join(word) for word in stemming_text]
#print the words after join
join_clean_text[:]

['from mamatha devineni ratnam andrewcmuedu subject pen fan reaction organ post offic carnegi mellon pittsburgh pa line nntppostinghost i am sure some basher of pen fan are pretti confus about the lack of ani kind of post about the recent pen massacr of the devil actual i am bit puzzl too and a bit reliev howev i am go to put an end to nonpittsburgh relief with a bit of prais for the pen man they are kill those devil wors than i thought jagr just show you whi he is much better than hi regular season stat he is also a lot fo fun to watch in the playoff bowman should let jagr have a lot of fun in the next coupl of game sinc the pen are go to beat the pulp out of jersey anyway i wa veri disappoint not to see the island lose the final regular season game pen rule',
 'from mblawson midwayecnuoknoredu matthew b lawson subject which highperform vlb video card summari seek recommend for vlb video card nntppostinghost midwayecnuoknoredu organ engin comput network univers of oklahoma norman ok u

## **white space**

In [281]:
# Final preprocessing step: remove leading and trailing white spaces from each document
final_clean_docs = [doc.strip() for doc in join_clean_text]
final_clean_docs[:]

['from mamatha devineni ratnam andrewcmuedu subject pen fan reaction organ post offic carnegi mellon pittsburgh pa line nntppostinghost i am sure some basher of pen fan are pretti confus about the lack of ani kind of post about the recent pen massacr of the devil actual i am bit puzzl too and a bit reliev howev i am go to put an end to nonpittsburgh relief with a bit of prais for the pen man they are kill those devil wors than i thought jagr just show you whi he is much better than hi regular season stat he is also a lot fo fun to watch in the playoff bowman should let jagr have a lot of fun in the next coupl of game sinc the pen are go to beat the pulp out of jersey anyway i wa veri disappoint not to see the island lose the final regular season game pen rule',
 'from mblawson midwayecnuoknoredu matthew b lawson subject which highperform vlb video card summari seek recommend for vlb video card nntppostinghost midwayecnuoknoredu organ engin comput network univers of oklahoma norman ok u

# **TF-IDF Process**

In [282]:
#from sklearn import TfidfVectorizer class
from sklearn.feature_extraction.text import TfidfVectorizer
# create an object from the class to convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer()
# Fit the vectorizer on the cleaned documents and transform them into a TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(final_clean_docs)
# Print the shape of the TF-IDF matrix
print(tfidf_matrix.shape)

(30, 2128)


In [283]:
# extract all  unique terms  learned by the vectorizer
#vocabulary = vectorizer.get_feature_names_out()
#Display the extracted vocabulary terms
#print(vocabulary)

## **User input**

In [284]:
# user search
#query_user= "football championship results fd5s 165 aedd1 is are          is @# "
query_user= input("Enter your search: ")
# convert the user query to string type
query=str(query_user)
# tokenize text into words
tokens_query = word_tokenize(query)
# lower case every term
lower_query =[word.lower() for word in tokens_query]
# remove stop words
stop_word_query_del=[word for word in lower_query if word not in stop_words]
# remove punctuation, numbers, numbers with words
punctuation_query_del= [word for word in stop_word_query_del if word not in punctuation_text and not word.isdigit()  and not any (num_word.isdigit() for num_word in word)]
# remove special characters
remove_signs_query = [re.sub(r'[^a-zA-Z\s]', '', word) for word in punctuation_query_del]
# convert every term to its root
stemming_query=[stemmer.stem(word) for word in remove_signs_query]
# join words together into text
join_clean_query=[' '.join(stemming_query)]
# remove white spaces
final_clean_query = [doc.strip() for doc in join_clean_query]
# convert text user query into TF-IDF  vector
query_vec = vectorizer.transform(final_clean_query)
# print the query after processed
print(final_clean_query)
# display the query TF-IDF vector
print(query_vec)

Enter your search: football championship results fd5s 165 aedd1 is are          is @# 
['footbal championship result']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 2128)>
  Coords	Values
  (0, 313)	0.7071067811865475
  (0, 1565)	0.7071067811865475


## **Cosine Similarity**

In [285]:
# import cosine_similarity class form sklearn to measure similarity between vectors
from sklearn.metrics.pairwise import cosine_similarity
#  calculate cosine similarity scores between the user query vector and all document vectors
similarities = cosine_similarity(query_vec, tfidf_matrix)
# display similarity scores between the query and each document
print(similarities)

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.04085767 0.         0.03748825 0.         0.         0.        ]]


In [286]:
# ranking documents in descending order based on  cosine similarity scores
ranking = np.argsort(similarities[0])[::-1]
# print the ranked documents from most relevant to least relevant
print(ranking)

[24 26 28 29 27 25 23 22 21 20 19 18 17 16 15 14 13 12 11 10  9  8  7  6
  5  4  3  2  1  0]


In [287]:
#disply the top 5 documents with the highest cosine similarit
top_5_docs = ranking[:5]
# display the indices of these top 5 most relevant documents
print("Top 5 relevant documents indices:", top_5_docs)

Top 5 relevant documents indices: [24 26 28 29 27]


In [288]:
# print the  top 5 documents with the  cosine similarit scores
for idx in top_5_docs:
    print(f"Score: {similarities[0][idx]:.4f} | Document: {text[idx]}")



Score: 0.0409 | Document: From: cs902043@ariel.yorku.ca (SHAWN LUDDINGTON)
Subject: Re: 1993 NHL Draft
Organization: York University, Toronto, Canada
Lines: 51

In article <93109.134719IO91748@MAINE.MAINE.EDU> Jon Carr <IO91748@MAINE.MAINE.EDU> writes:
>When is the draft this year?  And will there be any coverage?
>I know the upcomming NFL draft is on ESPN.
>
>Anyone got the details?
>
>Paul Kariya 1993 #1 Pick! (No. 2 perhaps?  He won't last long!) :-)
>
I don't know the exact coverage in the states.  In Canada it is covered
by TSN, so maybe ESPN will grab their coverage!  I don't know!

As for the picks
Ottawa picks #1 which means it is almost 100% that Alexander Daigle will 
go #1. He'll either stay or be traded in Montreal or Quebec.  IMO I would
take Kariya.  He should alot of leadership in the NCAA and so far in
the World Championships.  Daigle didn't show this for his junior team.

San Jose will then get Kariya.

Tampa Bay will either go for a russian Kozlov (I think that's it) 